# MoA(Mixture of Agents)로 금융 조사 보고서 자동 생성하기

이번 실습에서는 **MoA(Mixture of Agents)** 패턴을 활용하여 **금융 조사 서비스 백엔드**를 구현합니다.  
3개의 서로 다른 관점의 에이전트가 **EXA API로 실시간 웹 검색**을 수행하고, Aggregator가 이를 **하나의 통합 보고서**로 종합합니다.

> **Mixture of Agents + EXA 웹 검색**
> 
> 서로 다른 관점의 에이전트가 병렬로 EXA 웹 검색을 수행하고, 결과를 종합하는 패턴입니다.
> - **시장 분석 에이전트**: 시장 동향, 주가, 트렌드 검색
> - **리스크 분석 에이전트**: 리스크 요인, 규제, 위협 검색
> - **기회 탐색 에이전트**: 투자 기회, 성장 동력, 혁신 검색
> - **Aggregator**: 3개의 조사 결과를 통합 보고서로 작성

### 학습 목표
1. LangGraph에서 병렬 노드 실행 (Fan-out/Fan-in) 구현하기
2. EXA API를 활용한 실시간 웹 검색을 에이전트에 통합하기
3. 여러 검색 결과를 종합하는 Aggregator로 통합 보고서 생성하기

## 목차

1. [환경 설정](#1-환경-설정)
2. [EXA 검색 도구 및 상태 정의](#2-exa-검색-도구-및-상태-정의)
3. [관점별 리서치 에이전트 구현](#3-관점별-리서치-에이전트-구현)
4. [통합 보고서 Aggregator 구현](#4-통합-보고서-aggregator-구현)
5. [MoA 그래프 구성](#5-moa-그래프-구성)
6. [실행 및 테스트](#6-실행-및-테스트)

---
## 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
%pip install -qU langchain langgraph langchain-openai python-dotenv exa_py

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 설정
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

if not os.getenv("EXA_API_KEY"):
    os.environ["EXA_API_KEY"] = "your-exa-api-key"

print("환경 설정 완료!")
print("   OPENAI_API_KEY:", "설정됨" if os.getenv("OPENAI_API_KEY") != "your-openai-api-key" else "미설정")
print("   EXA_API_KEY:", "설정됨" if os.getenv("EXA_API_KEY") != "your-exa-api-key" else "미설정")

---
## 2. EXA 검색 도구 및 상태 정의

EXA API 클라이언트를 초기화하고, 각 에이전트가 공통으로 사용할 웹 검색 함수와 MoA 상태를 정의합니다.

In [ ]:
from typing import TypedDict, Annotated
import operator
from exa_py import Exa
from langchain.tools import tool

# EXA 클라이언트 초기화
exa_client = Exa(api_key=os.environ["EXA_API_KEY"])

@tool
def exa_search(query: str, num_results: int = 5) -> str:
    """
    EXA API를 사용하여 웹 검색을 수행합니다.
    각 에이전트가 자신의 관점에 맞는 검색 쿼리로 호출합니다.
    """
    results = exa_client.search_and_contents(
        query=query,
        num_results=num_results,
        text={"max_characters": 1000},
        highlights={"num_sentences": 3}
    )
    
    output = []
    for r in results.results:
        section = f"[{r.title}]\n   URL: {r.url}"
        if r.highlights:
            section += f"\n   핵심: {' '.join(r.highlights)}"
        if hasattr(r, 'text') and r.text:
            section += f"\n   본문: {r.text[:1000]}..."
        output.append(section)
    
    return "\n\n".join(output) if output else "검색 결과 없음"

# 상태 정의
class MoAState(TypedDict, total=False):
    """금융 조사 MoA 상태 - 3개 관점별 리포트가 별도 필드에 저장됨"""
    user_input: str  # 조사 주제 (예: "삼성전자 반도체 사업 전망")
    market_report: str  # 시장 분석 에이전트의 리포트
    risk_report: str    # 리스크 분석 에이전트의 리포트
    opportunity_report: str  # 기회 탐색 에이전트의 리포트
    final_answer: str   # Aggregator가 작성한 통합 보고서
# responses 필드는 삭제 (개별 관점 리포트 별도 저장용 필드 도입)

In [ ]:
print("EXA 검색 도구 및 상태 정의 완료!")

# 간단한 검색 테스트
test = exa_search.invoke("2025 글로벌 반도체 시장 전망", num_results=2)

In [ ]:
print(f"\nEXA 검색 테스트:\n{test[:3000]}")

---
## 3. 관점별 리서치 에이전트 구현

금융 조사를 위한 3개의 서로 다른 관점의 에이전트를 구현합니다.  
각 에이전트는 **EXA API로 웹 검색을 수행**한 뒤, LLM이 검색 결과를 바탕으로 해당 관점의 분석 리포트를 작성합니다.

| 에이전트 | 역할 | 검색 키워드 예시 |
|----------|------|------------------|
| 시장 분석 | 시장 동향, 주가, 실적, 점유율 | `"삼성전자 반도체 시장점유율 2025"` |
| 리스크 분석 | 규제, 지정학 리스크, 경쟁 위협 | `"반도체 수출 규제 리스크"` |
| 기회 탐색 | 신기술, 투자 기회, 성장 동력 | `"AI 반도체 투자 기회 성장"` |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware, ToolRetryMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from datetime import datetime
# 여기서 exa_search 툴만 1초에 5회(rps=5) 제한하도록 설정
exa_tool_limit_middleware = ToolCallLimitMiddleware(
    tool_name="exa_search",
    thread_limit=5,   # 동시에 5개까지
    run_limit=5       # 최근 1초 내 최대 5번까지 호출 허용 (rps 제어)
)

# Retry 미들웨어도 추가 (최대 3회, backoff factor 2.0, 초기 지연 1초)
exa_tool_retry_middleware = ToolRetryMiddleware(
    max_retries=3,
    backoff_factor=2.0,
    initial_delay=1.0
)

today = datetime.today().strftime('%Y-%m-%d')
date_prompt=f"""
!!!!! 반드시 주의 !!!!!
[중요] 오늘 날짜는 반드시 {today}입니다. 
절대로 이전 연도(2023/2024 등)가 아닌, {today} 기준으로만 분석하세요.
날짜를 임의로 유추해서 쓰지 말 것.\n"""

# 시장 분석 에이전트 세팅
market_system_prompt = date_prompt+f"""
당신은 주어진 기업이 속한 산업 분석 전문가입니다.
exa_search로 받은 검색 결과를 바탕으로 입력 주제에 대해 아래와 같이 산업 분석 리포트를 작성하세요.

- 현재 시장 규모 및 동향
- 주요 플레이어 및 시장 점유율
- 최근 실적 및 재무 지표
- 시장 전망 (단기/중기)

검색 결과에 기반하여 팩트 중심으로 한국어로 작성하세요."""

_market_agent_core = create_agent(
    model="gpt-4.1-mini",
    tools=[exa_search],
    system_prompt=market_system_prompt,
    middleware=[
        exa_tool_limit_middleware,
        exa_tool_retry_middleware
    ]
)

def market_agent(state: MoAState) -> dict:
    """
    MoAState(state)를 받아 '산업 분석' 관점의 보고서를 작성하여 dict 반환.
    """
    user_input = state['user_input']
    message= [HumanMessage(content=f"{user_input}")]
    result = _market_agent_core.invoke({"messages": message})
    content = result['messages'][-1].content
    return {"market_report": content} 

# 리스크 분석 에이전트 세팅
risk_system_prompt = date_prompt+f"""
오늘은 {today}입니다.
당신은 주어진 기업의 리스크 분석 전문가입니다.
exa_search로 받은 검색 결과를 바탕으로 기업의 리스크 분석 리포트를 작성하세요.

- 주요 리스크 요인 (규제, 지정학, 거시경제 등)
- 경쟁 위협 및 시장 포지션 변화 가능성
- 기술적 리스크 (공급망, 기술 전환 등)
- 리스크 심각도 평가 (높음/중간/낮음)

검색 결과에 기반하여 팩트 중심으로 한국어로 작성하세요."""

_risk_agent_core = create_agent(
    model="gpt-4.1-mini",
    tools=[exa_search],
    system_prompt=risk_system_prompt,
    middleware=[
        exa_tool_limit_middleware,
        exa_tool_retry_middleware
    ]
)

def risk_agent(state: MoAState) -> dict:
    """
    MoAState(state)를 받아 '리스크 분석' 관점의 보고서를 작성하여 dict 반환.
    """
    user_input = state['user_input']
    message= [HumanMessage(content=f"{user_input}")]
    result = _risk_agent_core.invoke({"messages": message})
    content = result['messages'][-1].content
    return {"risk_report": content}

# 기회 탐색 에이전트 세팅
opportunity_system_prompt = date_prompt+f"""
오늘은 {today}입니다.
당신은 주어진 기업의 기회 분석 전문가입니다.
exasearch_tool로 받은 검색 결과를 바탕으로 주어진 기업의 기회 리포트를 작성하세요.

- 핵심 성장 동력 및 촉매 요인
- 신기술/신사업 기회
- 투자 관점에서의 매력도 평가
- 향후 기대되는 이벤트 및 카탈리스트

검색 결과에 기반하여 팩트 중심으로 한국어로 작성하세요."""

_opportunity_agent_core = create_agent(
    model="gpt-4.1-mini",
    tools=[exa_search],
    system_prompt=opportunity_system_prompt,
    middleware=[
        exa_tool_limit_middleware,
        exa_tool_retry_middleware
    ]
)

def opportunity_agent(state: MoAState) -> dict:
    """
    MoAState(state)를 받아 '기회 탐색' 관점의 보고서를 작성하여 dict 반환.
    """
    user_input = state['user_input']
    message= [HumanMessage(content=f"{user_input}")]
    result = _opportunity_agent_core.invoke({"messages": message})
    content = result['messages'][-1].content
    return {"opportunity_report": content}

print("3개 MoAState 기반 리서치 에이전트 함수 구현 완료!")
print("   [시장 분석] market_agent")
print("   [리스크 분석] risk_agent")
print("   [기회 탐색] opportunity_agent")

# Aggregator (통합): 각 에이전트의 결과(텍스트)를 state로 받아 synthesize
def aggregator(state: MoAState) -> dict:
    market_report = state['market_report']
    risk_report = state['risk_report']
    opportunity_report = state['opportunity_report']

    user_input = state["user_input"]
    llm = ChatOpenAI(model="gpt-4.1", temperature=0.2)

    inputs = (
        f"# [{user_input}] 통합 조사 보고서\n\n"
        "## 1. Executive Summary\n"
        "(핵심 요약 - 3~5줄)\n\n"
        "## 2. 시장 현황 및 동향\n"
        f"{market_report}\n\n"
        "## 3. 리스크 요인\n"
        f"{risk_report}\n\n"
        "## 4. 투자 기회 및 성장 동력\n"
        f"{opportunity_report}\n\n"
        "## 5. 종합 평가 및 전망\n"
        "(3개 관점 종합 최종 의견 제시)\n"
    )
    prompt = (
        "당신은 기업 리서치 편집장입니다.\n"
        "아래 세 가지 관점(시장, 리스크, 기회)별 리포트 전문을 보고 구조화된 통합 금융 조사 보고서 최종본(#머리말/Executive/시장현황/리스크/기회/종합평가)을 한국어로 완성하세요.\n"
        "중복은 통합, 충돌시 양쪽 모두 언급. 핵심 명확하게 요약.\n\n"
        "=== 원본 ===\n"
        f"{inputs}\n"
        "=== 원본 끝 ===\n\n"
        f"# [{user_input}] 통합 조사 보고서:"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content

    return {"final_answer": content}

---
## 4. 통합 보고서 Aggregator 구현

3개 에이전트의 조사 결과(시장 분석 / 리스크 분석 / 기회 탐색)를 종합하여  
**하나의 구조화된 금융 조사 보고서**를 생성합니다.

---
## 5. MoA 그래프 구성

Fan-out/Fan-in 패턴으로 3개 에이전트를 **병렬 실행**한 뒤 Aggregator로 수렴합니다.
```
              +-> market_research_agent ---+
START --------+-> risk_research_agent -----+-> aggregator -> END
              +-> opportunity_research_agent+
```

In [ ]:
from langgraph.graph import StateGraph, START, END

# 그래프 생성
workflow = StateGraph(MoAState)

# 노드 추가
workflow.add_node("market_research", market_agent)
workflow.add_node("risk_research", risk_agent)
workflow.add_node("opportunity_research", opportunity_agent)
workflow.add_node("aggregator", aggregator)

# Fan-out: START에서 3개 리서치 에이전트로 분기 (병렬 실행)
workflow.add_edge(START, "market_research")
workflow.add_edge(START, "risk_research")
workflow.add_edge(START, "opportunity_research")

# Fan-in: 3개 에이전트에서 aggregator로 수렴
workflow.add_edge("market_research", "aggregator")
workflow.add_edge("risk_research", "aggregator")
workflow.add_edge("opportunity_research", "aggregator")

# aggregator에서 종료
workflow.add_edge("aggregator", END)

# 컴파일
moa_agent = workflow.compile()

print("MoA 금융 조사 워크플로우 컴파일 완료!")

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

try:
    display(Image(moa_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"시각화 오류: {e}")
    print("\n그래프 구조:")
    print("START -> [market_research / risk_research / opportunity_research] -> aggregator -> END")

---
## 6. 실행 및 테스트

실제 금융 조사 주제를 입력하여 MoA 파이프라인을 실행합니다.  
3개 에이전트가 **각각 EXA로 웹 검색**을 수행한 뒤, Aggregator가 통합 보고서를 생성합니다.

In [ ]:
# 금융 조사 주제 설정
question = "삼성전자"

print(f"조사 주제: {question}")
print("=" * 70)
print("3개 에이전트가 병렬로 EXA 웹 검색을 수행합니다...\n")

result = moa_agent.invoke({"user_input": question})

In [ ]:
print(result['final_answer'])

---
## 정리

### MoA + EXA 웹 검색 패턴의 장점

| 장점 | 설명 |
|------|------|
| **실시간 정보** | EXA API로 최신 웹 데이터 기반 분석 |
| **다관점 분석** | 시장/리스크/기회 3가지 축으로 균형 잡힌 조사 |
| **병렬 처리** | 3개 에이전트가 동시에 검색하여 시간 효율적 |
| **구조화된 보고서** | Aggregator가 통합 보고서 형식으로 자동 생성 |

### 핵심 아키텍처
```
사용자 질문 (예: "삼성전자 반도체 전망")
        |
   +----+----------------+
   |    |                |
   v    v                v
 시장분석  리스크분석     기회탐색
 EXA검색  EXA검색       EXA검색
 LLM분석  LLM분석       LLM분석
   |    |                |
   +----+----------------+
        v
   Aggregator
   (통합 보고서 생성)
```

### 실제 프로덕션 적용 시 고려사항
- EXA API 키 관리 및 요금제 확인
- 에이전트 수 x 검색 횟수에 따른 비용 최적화
- EXA 검색 + LLM 호출 응답 시간 관리 (타임아웃 설정)
- 도메인별 검색 쿼리 최적화 (금융, 의료, 법률 등)
- 검색 결과 캐싱으로 동일 주제 재조사 비용 절감

### 다음 단계
**CH02._04.** CodeAct 에이전트로 복잡한 문제 해결하기